In [4]:
# fqn

# serialize
# attribute list
# get attr


# deserialize
# obj = class_type.__new__(class_type)  # type: ignore
# for attr_name, attr_value in kwargs.items():
    # setattr(obj, attr_name, attr_value)

# support unknown data types

In [5]:
import warnings

# Ignore DeprecationWarning
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [6]:
from dataclasses import dataclass

In [7]:
from typing import Any

In [167]:
class InMemory(DataAttrs, A):
    pass

In [8]:
@dataclass
class DataAttrs:
    keys: list[str]
    values: list[Any]
    fqn: str
    leaf_blob: bytes | None

In [9]:
class A:
    a: int
    pass

In [ ]:
@dataclass
class DataAttrs:
    types: list[ReduxType]
    keys: list[str]
    values: list[Any]
    fqn: str
    leaf_blob: bytes | None

In [10]:
@dataclass
class Thing:
    a: int
    b: str

In [11]:
t = Thing(a=1, b="test")

In [12]:
t.__dict__.keys()

dict_keys(['a', 'b'])

In [13]:
from typing import Any

In [14]:
def is_builtin_function_or_method(obj):
    return type(obj).__name__ == 'builtin_function_or_method'

In [15]:
def get_fqn_obj(obj: Any) -> str:
    fqn = obj.__class__.__module__

    try:
        fqn += "." + obj.__class__.__name__
    except Exception as e:
        print(e)
    return fqn

In [16]:
def get_fqn_t_or_f(t_or_f: Any) -> str:
    fqn = t_or_f.__module__

    try:
        fqn += "." + t_or_f.__name__
    except Exception as e:
        print(e)
    return fqn

In [17]:
def is_type(t_or_o) -> bool:
    if type(t_or_o) is type or is_builtin_function_or_method(t_or_o):
        return True
    return False

In [18]:
def get_fqn(t_or_o: type | object) -> str:
    if is_type(t_or_o):
        return get_fqn_t_or_f(t_or_o)
    return get_fqn_obj(t_or_o)

In [20]:
import types

# Example objects
builtin_function = len
user_defined_function = lambda x: x
builtin_method = [].append
regular_method = list().append

# Check if the objects are built-in functions or methods
print(isinstance(builtin_function, types.BuiltinFunctionType))  # True
print(isinstance(user_defined_function, types.BuiltinFunctionType))  # False
print(isinstance(builtin_method, types.BuiltinMethodType))  # True
print(isinstance(regular_method, types.BuiltinMethodType))  # False

# To handle both built-in functions and methods, you can combine the checks
def is_builtin_function_or_method(obj):
    return isinstance(obj, (types.BuiltinFunctionType, types.BuiltinMethodType))

print(is_builtin_function_or_method(builtin_function))  # True
print(is_builtin_function_or_method(user_defined_function))  # False
print(is_builtin_function_or_method(builtin_method))  # True
print(is_builtin_function_or_method(regular_method))  # False


True
False
True
True
True
False
True
True


In [27]:
fqn = get_fqn(t)
fqn

'__main__.Thing'

In [28]:
TYPE_BANK = {}

In [29]:
TYPE_BANK[fqn] = (Thing, )

In [30]:
def serialize(obj: Any) -> DataAttrs:
    fqn = get_fqn(obj)
    print(fqn)
    parts = TYPE_BANK[fqn]
    t = parts[0]
    attrs = obj.__dict__.keys()
    values = []
    for attr in attrs:
        print(attr)
        values.append(getattr(obj, attr))
    d = DataAttrs(keys=attrs, values=values, fqn=fqn, leaf_blob=None)
    return d

In [31]:
d = serialize(t)

__main__.Thing
a
b


In [32]:
d

DataAttrs(keys=dict_keys(['a', 'b']), values=[1, 'test'], fqn='__main__.Thing', leaf_blob=None)

In [33]:
getattr(t, "a")

1

In [34]:
def deserialize(data: DataAttrs) -> Any:
    fqn = data.fqn
    parts = TYPE_BANK[fqn]
    t = parts[0]
    obj = t.__new__(t)
    for k, v in zip(data.keys, data.values):
        print(k, v)
        setattr(obj, k, v)
    return obj

In [35]:
d = serialize(t)

__main__.Thing
a
b


In [36]:
d

DataAttrs(keys=dict_keys(['a', 'b']), values=[1, 'test'], fqn='__main__.Thing', leaf_blob=None)

In [37]:
e = deserialize(d)

a 1
b test


In [38]:
e == t

True

In [39]:
import numpy as np

In [40]:
x = np.array([1, 2, 3])

In [41]:
x

array([1, 2, 3])

In [42]:
x.shape[0]

3

In [43]:
x


array([1, 2, 3])

In [44]:
ignore_list = ["__doc__", "__module__", "__weakref__"]
def decompose_object(obj):
    attributes = dir(obj)
    values = []
    keys = []
    for attr in attributes:
        if attr in ignore_list:
            continue
        try:
            value = getattr(obj, attr)
            if isinstance(obj.__class__.__dict__.get(attr), property) or not callable(value):
                values.append(value)
                keys.append(attr)
        except AttributeError as e:
            # Handle the case where an attribute might not be accessible
            print(attr, e)

    return keys, values, type(obj)

In [45]:
# Example usage
class Example:
    def __init__(self):
        self.a = 1
        self.b = "hello"
        self.c = [1, 2, 3]
    
    @property
    def prop(self):
        return "This is a property"
    
    def method(self):
        return "This is a method"

example_obj = Example()
decomposed_values = decompose_object(example_obj)
print(decomposed_values)

(['__dict__', 'a', 'b', 'c', 'prop'], [{'a': 1, 'b': 'hello', 'c': [1, 2, 3]}, 1, 'hello', [1, 2, 3], 'This is a property'], <class '__main__.Example'>)


In [46]:
decomposed_values

(['__dict__', 'a', 'b', 'c', 'prop'],
 [{'a': 1, 'b': 'hello', 'c': [1, 2, 3]},
  1,
  'hello',
  [1, 2, 3],
  'This is a property'],
 __main__.Example)

In [47]:
def has_setter(obj, attr_name):
    """
    Checks if a given property of an object has a setter.

    Args:
    - obj: The object to check.
    - attr_name: The name of the attribute to check.

    Returns:
    - True if the property has a setter, False otherwise.
    """
    attr = getattr(obj.__class__, attr_name, None)
    return isinstance(attr, property) and attr.fset is not None

In [48]:
def recompose_object(obj_type, keys, values, new_kwargs = None):
    """
    Reconstructs an object of a given type from a list of values.
    
    Args:
    - obj_type: The type of the object to reconstruct.
    - values: A list of values corresponding to the object's attributes.
    
    Returns:
    - The reconstructed object.
    """
    if new_kwargs is None:
        new_kwargs = {}
    # Create a new instance of the given type without calling __init__
    obj = obj_type.__new__(obj_type, **new_kwargs)
    
    # Get the attributes of the object
    attributes = dir(obj)
    
    # Filter out attributes that are not user-defined (skip built-in attributes)
    user_defined_attributes = [attr for attr in attributes if not attr.startswith('__')]

    # Assign the values to the corresponding attributes, ignoring non-property callables
    for attr, value in zip(keys, values):
        if isinstance(obj.__class__.__dict__.get(attr), property) and not has_setter(obj, attr):
            print(attr, "no setter, skipping")
            continue
        try:
            setattr(obj, attr, value)
        except AttributeError as e:
            if "is not writable" in str(e):
                continue
            elif "is read-only" in str(e):
                continue
            else:
                raise e
        except Exception as e:
            if "does not have" in str(e):
                continue
            print("failed with attr", attr)
            raise e
    
    return obj

In [49]:
# Example usage
d_keys, d_values, t = decompose_object(example_obj)
d_keys, d_values, t

(['__dict__', 'a', 'b', 'c', 'prop'],
 [{'a': 1, 'b': 'hello', 'c': [1, 2, 3]},
  1,
  'hello',
  [1, 2, 3],
  'This is a property'],
 __main__.Example)

In [50]:
recomposed_obj = recompose_object(t, d_keys, d_values)
recomposed_obj

prop no setter, skipping


In [51]:
print(recomposed_obj.a)    # Output: 1
print(recomposed_obj.b)    # Output: hello
print(recomposed_obj.c)    # Output: [1, 2, 3]
print(recomposed_obj.prop) # Output: This is a property

1
hello
[1, 2, 3]
This is a property


In [52]:
x

array([1, 2, 3])

In [53]:
d_keys, d_values, t = decompose_object(x)

In [54]:
d_keys, d_values, t

(['T',
  '__array_interface__',
  '__array_priority__',
  '__array_struct__',
  '__hash__',
  'base',
  'ctypes',
  'data',
  'dtype',
  'flags',
  'flat',
  'imag',
  'itemsize',
  'nbytes',
  'ndim',
  'real',
  'shape',
  'size',
  'strides'],
 [array([1, 2, 3]),
  {'data': (105553143089920, False),
   'strides': None,
   'descr': [('', '<i8')],
   'typestr': '<i8',
   'shape': (3,),
   'version': 3},
  0.0,
  <capsule object NULL at 0x10a142d30>,
  None,
  None,
  dtype('int64'),
    C_CONTIGUOUS : True
    F_CONTIGUOUS : True
    OWNDATA : True
    WRITEABLE : True
    ALIGNED : True
    WRITEBACKIFCOPY : False,
  array([0, 0, 0]),
  8,
  24,
  1,
  array([1, 2, 3]),
  (3,),
  3,
  (8,)],
 numpy.ndarray)

In [55]:
d_keys[16]

'shape'

In [56]:
d_values[16]

(3,)

In [57]:
new_kwargs = {d_keys[16]: d_values[16]}

In [58]:
recomposed_obj = recompose_object(t, d_keys, d_values, new_kwargs)
recomposed_obj

array([1, 2, 3])

In [59]:
recomposed_obj

array([1, 2, 3])

In [60]:
recomposed_obj == x

array([ True,  True,  True])

In [61]:
hasattr(x, "__reduce__")

True

In [62]:
import inspect

In [63]:
# inspect.getsource(x.__reduce__)

In [64]:
l = x.__reduce__()

In [65]:
len(l)

3

In [66]:
l[0]

<function numpy.core.multiarray._reconstruct>

In [67]:
l[1]

(numpy.ndarray, (0,), b'b')

In [68]:
l[2]

(1,
 (3,),
 dtype('int64'),
 False,
 b'\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00')

In [69]:
cls, args, state = x.__reduce__()

In [70]:
hash(type(1))

272257456

In [71]:
hash(type([]))

-9223372036582518493

In [72]:
hash(type([]))

-9223372036582518493

In [161]:
from collections.abc import Iterable

ts = set()

def is_iterable(obj):
    """Check if an object is iterable, but not a string."""
    return isinstance(obj, Iterable) and not isinstance(obj, (str, bytes, bytearray))

def recursively_iterate(collection):
    """Recursively iterate through all elements in a collection."""
    for item in collection:
        print("got item", item, type(item))
        ts.add(type(item))
        if is_type(item):
            l = ReduxType.from_obj(item)
            print(l)
        if is_iterable(item):
            print(f"Entering iterable: {item}")
            recursively_iterate(item)
        else:
            print(f"Processing item: {item}")

In [166]:
recursively_iterate(args)

got item <class 'numpy.ndarray'> <class 'type'>
ReduxType(package='numpy', version='1.26.4', fqn='numpy.ndarray')
Processing item: <class 'numpy.ndarray'>
got item (0,) <class 'tuple'>
Entering iterable: (0,)
got item 0 <class 'int'>
Processing item: 0
got item b'b' <class 'bytes'>
Processing item: b'b'


In [ ]:
class :


In [75]:
def serialize(obj: Any) -> bytes:
    primitives = (float, int, str, bytes)
    if isinstance(obj, primitives):
        import pickle
        b = pickle.dumps(obj)
        return b
    reduce()

In [76]:
serialize(1)

b'\x80\x04K\x01.'

In [77]:
def deserialize(data: bytes) -> Any:
    import pickle
    try:
        return pickle.loads(data)
    except Exception as e:
        print("failed")
        raise e

In [78]:
deserialize(serialize(1))

1

In [79]:
r = Redux.from_obj(x)
r

NameError: name 'Redux' is not defined

In [80]:
serialize(x)

NameError: name 'reduce' is not defined

In [95]:
x

array([1, 2, 3])

In [96]:
x

array([1, 2, 3])

In [97]:
x.__reduce__()

(<function numpy.core.multiarray._reconstruct>,
 (numpy.ndarray, (0,), b'b'),
 (1,
  (3,),
  dtype('int64'),
  False,
  b'\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00'))

In [82]:
ts

set()

In [100]:
from typing import Self

In [168]:
# p = ReduxType(web_did="openmined.org/types/user.json")

In [ ]:
# def serde_protocol_capnp():
# def serde_protocol_protobuf3():
# def serde_protocol_json():

In [146]:
@dataclass
class ReduxType:
    package: str
    version: str
    fqn: str
    # code: str | None
    # signature: str
    # signing_key: str
    # web_did: str

    def from_obj(obj: Any) -> Self:
        if not is_type(obj):
            obj = type(obj)
        cls = obj
        return ReduxType(
            package=get_package_name(cls),
            version=get_package_version(cls),
            fqn=get_fqn(cls),
        )

In [147]:
@dataclass
class Redux:
    constructor_fqn: ReduxType
    init_args: list[Any]
    state: list[Any]

    def from_obj(obj: Any) -> Self:
        cls, args, state = obj.__reduce__()
        rt = ReduxType.from_obj(cls)
        return Redux(    
            constructor_fqn=rt,
            init_args=args,
            state=state
        )

    def to_obj(self) -> Any:
        cls = dynamic_import(self.constructor_fqn.fqn)
        reconstructed_obj = cls(*self.init_args)
        reconstructed_obj.__setstate__(self.state)
        return reconstructed_obj

In [148]:
def get_package_name(obj):
    # Get the module name where the object is defined
    module_name = obj.__module__
    
    # Use importlib to import the module dynamically
    module = importlib.import_module(module_name)
    
    # Check if the module has a __package__ attribute
    package_name = getattr(module, '__package__', None)
    
    if package_name is None:
        # If __package__ is None, use the module name itself
        package_name = module_name
    
    return package_name

In [149]:
def get_package_version(obj):
    # Get the module name where the object is defined
    module_name = obj.__module__.split(".")[0]
    
    # Use importlib to import the module dynamically
    module = importlib.import_module(module_name)
    
    # Check if the module has a __package__ attribute
    package_name = getattr(module, '__package__', None)
    
    if package_name is None:
        # If __package__ is None, use the module name itself
        package_name = module_name
    
    try:
        # Get the package version using importlib.metadata
        package_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        # Handle the case where the package is not found
        package_version = 'Unknown'
    
    return package_version

In [150]:
get_package_version(cls)

'1.26.4'

In [151]:
from typing import Self

In [152]:
cls, args, state = x.__reduce__()

In [169]:
args

(numpy.ndarray, (0,), b'b')

In [173]:
args

(numpy.ndarray, (0,), b'b')

In [172]:
cls

<function numpy.core.multiarray._reconstruct>

In [158]:
r = Redux.from_obj(x)
r

Redux(constructor_fqn=ReduxType(package='numpy.core', version='1.26.4', fqn='numpy.core.multiarray._reconstruct'), init_args=(<class 'numpy.ndarray'>, (0,), b'b'), state=(1, (3,), dtype('int64'), False, b'\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00'))

In [159]:
y = r.to_obj()

In [160]:
y

array([1, 2, 3])

In [157]:
# r = Redux(
#     package=get_package_name(cls),
#     version=get_package_version(cls),
#     constructor_fqn=get_fqn(cls),
#     init_args=args,
#     state=state
# )

In [142]:
r

Redux(constructor_fqn=ReduxType(package='numpy.core', version='1.26.4', fqn='numpy.core.multiarray._reconstruct'), init_args=(<class 'numpy.ndarray'>, (0,), b'b'), state=(1, (3,), dtype('int64'), False, b'\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00'))

In [143]:
# Step 3: Manually reconstruct the object using the reconstruction data
# Reconstruct the object by calling the class with the arguments
reconstructed_obj = cls(*args)

In [144]:
reconstructed_obj

array([], dtype=int8)

In [145]:
# Restore the state
reconstructed_obj.__setstate__(state)

In [122]:
reconstructed_obj

array([1, 2, 3])

In [123]:
args

(numpy.ndarray, (0,), b'b')

In [124]:
type(cls)

builtin_function_or_method

In [125]:
from numpy.core.multiarray import _reconstruct

In [126]:
import importlib

def dynamic_import(full_path):
    """
    Dynamically import a module or function given its full path.
    
    :param full_path: Full path to the module or function, e.g., 'numpy.core.multiarray._reconstruct'
    :return: The imported module or function
    """
    module_path, _, function_name = full_path.rpartition('.')
    
    # Import the module dynamically
    module = importlib.import_module(module_path)
    
    if function_name:
        # Get the function from the module
        return getattr(module, function_name)
    else:
        # Return the module itself if no function is specified
        return module

# Example usage
imported_function = dynamic_import('numpy.core.multiarray._reconstruct')
print(imported_function)  # Output: <built-in function _reconstruct>

imported_module = dynamic_import('numpy.core.multiarray')
print(imported_module)  # Output: <module 'numpy.core.multiarray' from '...'>


<built-in function _reconstruct>
<module 'numpy.core.multiarray' from '/Users/madhavajay/dev/PySyft/.tox/syft.jupyter/.venv/lib/python3.12/site-packages/numpy/core/multiarray.py'>


In [127]:
y = get_fqn(imported_function)

In [128]:
dynamic_import(y)

<function numpy.core.multiarray._reconstruct>

In [129]:
type(imported_function)

builtin_function_or_method

In [130]:
imported_function.__name__

'_reconstruct'

In [131]:
get_fqn(imported_function)

'numpy.core.multiarray._reconstruct'

In [132]:
imported_function.__module__

'numpy.core.multiarray'

In [133]:
imported_function.__name__

'_reconstruct'